# Smart Farming — Notebook 3: AI4Bharat Indic-TTS Deployment

Runs **AI4Bharat Indic-TTS alone on port 8004** in a dedicated Google Colab runtime.

- **Models**: Separate language-specific FastPitch acoustic models + HiFi-GAN vocoders for **Indian English (`en`)**, **Gujarati (`gu`)**, and **Hindi (`hi`)**.
- **Hardware Acceleration**: Automatically detects and leverages **CUDA GPU (T4)** for fast ~200ms speech synthesis, while smoothly falling back to CPU if no GPU is available.
- **Persistent Drive Storage**: Checkpoint archives (`en.zip`, `gu.zip`, `hi.zip` ~4.2 GB total) are downloaded and extracted to Google Drive (`MyDrive/SmartFarming/models/indic_tts/checkpoints`) once, and copied to local SSD on startup to eliminate repeated downloads.
- **Cloudflare Integration**: Exposes port 8004 via Cloudflare Quick Tunnel and automatically updates the stable Cloudflare Worker Gateway's KV key `TTS_URL`.

**Prerequisites:** Add Colab Secrets `CF_API_TOKEN`, `CF_ACCOUNT_ID`, `CF_KV_NAMESPACE_ID`, `CF_WORKER_BASE_URL`, and `GATEWAY_API_KEY`.


## 1. Mount Drive and prepare directories


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess, time, signal, re, json, requests, shutil
from pathlib import Path

DRIVE_ROOT = "/content/drive/MyDrive/SmartFarming"
TTS_SERVER_DIR = "/content/drive/MyDrive/IndicTTSServer"
CHECKPOINTS_DRIVE = os.path.join(DRIVE_ROOT, "models", "indic_tts", "checkpoints")
LOCAL_CHECKPOINTS = "/content/checkpoints"

os.makedirs(TTS_SERVER_DIR, exist_ok=True)
os.makedirs(CHECKPOINTS_DRIVE, exist_ok=True)
os.makedirs(LOCAL_CHECKPOINTS, exist_ok=True)

print("Drive checkpoints dir:", CHECKPOINTS_DRIVE)
print("Local SSD checkpoints dir:", LOCAL_CHECKPOINTS)


## 2. Checkpoint management (Drive persistence & local SSD cache)

Downloads official AI4Bharat Indic-TTS v1 checkpoints for `en`, `gu`, and `hi` directly into Google Drive if not already present, then copies them to Colab's high-speed local disk.


In [ ]:
LANGUAGES = ["en", "gu", "hi"]
BASE_RELEASE_URL = "https://github.com/AI4Bharat/Indic-TTS/releases/download/v1-checkpoints-release"

def verify_model_files(folder):
    required = [
        os.path.join(folder, "fastpitch", "best_model.pth"),
        os.path.join(folder, "fastpitch", "config.json"),
        os.path.join(folder, "hifigan", "best_model.pth"),
        os.path.join(folder, "hifigan", "config.json"),
    ]
    return all(os.path.isfile(p) for p in required)

def flatten_if_nested(folder):
    """If a zip unpacked into a extra top-level folder (en/en/fastpitch/...), move its contents up one level."""
    if verify_model_files(folder) or not os.path.isdir(folder):
        return
    for root, dirs, _files in os.walk(folder):
        if "fastpitch" in dirs and "hifigan" in dirs:
            if os.path.abspath(root) != os.path.abspath(folder):
                for name in os.listdir(root):
                    shutil.move(os.path.join(root, name), os.path.join(folder, name))
                print(f"  Moved checkpoint files up from {root}")
            return

for lang in LANGUAGES:
    drive_lang_dir = os.path.join(CHECKPOINTS_DRIVE, lang)
    local_lang_dir = os.path.join(LOCAL_CHECKPOINTS, lang)

    flatten_if_nested(drive_lang_dir)
    if not verify_model_files(drive_lang_dir):
        print(f"\n[Download] Checkpoints for '{lang}' not found in Google Drive. Downloading from release...")
        zip_url = f"{BASE_RELEASE_URL}/{lang}.zip"
        temp_zip = f"/content/{lang}.zip"
        subprocess.run(["wget", "-c", "--show-progress", zip_url, "-O", temp_zip], check=True)
        print(f"Extracting {lang}.zip to {drive_lang_dir}...")
        os.makedirs(drive_lang_dir, exist_ok=True)
        subprocess.run(["unzip", "-q", "-o", temp_zip, "-d", drive_lang_dir], check=True)
        if os.path.isfile(temp_zip):
            os.remove(temp_zip)
        flatten_if_nested(drive_lang_dir)
        assert verify_model_files(drive_lang_dir), (
            f"'{lang}' checkpoints are incomplete after extraction. Look inside {drive_lang_dir} "
            "for fastpitch/ and hifigan/ folders.")
        print(f"[Done] Checkpoints for '{lang}' saved to Google Drive.")
    else:
        print(f"[Cached] Checkpoints for '{lang}' found in Google Drive.")

    # Fast copy from Drive to local SSD
    if not verify_model_files(local_lang_dir):
        print(f"Copying '{lang}' checkpoints to local SSD {local_lang_dir}...")
        shutil.copytree(drive_lang_dir, local_lang_dir, dirs_exist_ok=True)
    assert verify_model_files(local_lang_dir), f"'{lang}' checkpoints missing on local SSD {local_lang_dir}"
    print(f"Local SSD cache for '{lang}' ready.")

print("\nAll model checkpoints verified and ready on local SSD!")

## 3. Install system and Python dependencies


In [ ]:
# System libraries for audio processing & spellcheck/normalization
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "libsndfile1-dev", "ffmpeg", "enchant-2", "libenchant-2-dev"], check=True)

# Upgrade pip and packaging tools first to avoid build failures with legacy packages
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools", "wheel"], check=True)

# FastAPI and server stack
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "fastapi", "uvicorn[standard]", "requests", "soundfile", "scipy", "pydantic"], check=True)

# Text normalization and transliteration tools
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "indic-numtowords", "aksharamukha", "pyenchant", "pysbd"], check=True)

# Coqui TTS / Indic-TTS synthesis engine
# Installing from a maintained repository path or using pre-compiled wheels is standard for PyTorch 2.x and Python 3.10+
# To avoid dependency conflicts with numpy/numba, we install it with loose dependency checks or via github directly
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "coqui-tts"], check=True)

print("Indic-TTS dependencies installed successfully.")

## 4. Write Indic-TTS FastAPI server to Drive


In [ ]:
%%writefile /content/drive/MyDrive/IndicTTSServer/indic_tts_server.py
import io
import json
import os
import re
import base64
import threading
from pathlib import Path
import numpy as np
import soundfile as sf
import torch
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import Response
from pydantic import BaseModel, Field
from TTS.utils.synthesizer import Synthesizer

# --------------------------------------------------------------------------
# Hardware Configuration
# --------------------------------------------------------------------------
USE_CUDA = torch.cuda.is_available()
DEVICE = "cuda" if USE_CUDA else "cpu"
if not USE_CUDA:
    CPU_THREADS = int(os.getenv("TTS_CPU_THREADS", str(os.cpu_count() or 2)))
    torch.set_num_threads(CPU_THREADS)
print(f"[Init] Indic-TTS starting on device: {DEVICE} (CUDA available: {USE_CUDA})")

# Checkpoint folders
CHECKPOINTS = Path("/content/checkpoints")
LANGUAGES = {
    "en": "Indian English",
    "gu": "Gujarati",
    "hi": "Hindi",
}

app = FastAPI(title="AI4Bharat Indic-TTS Server", version="1.0.0")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# --------------------------------------------------------------------------
# Model Management
# --------------------------------------------------------------------------
# All three languages are loaded at startup (see preload_all at the bottom of this file).
_models = {}
_model_lock = threading.Lock()

def _find_file(folder: Path, name: str):
    hits = sorted(folder.rglob(name))
    return hits[0] if hits else None

def _localize_config(cfg_path: Path, folder: Path) -> Path:
    """Checkpoint configs can contain absolute paths from another machine (e.g. a Drive path to speakers.pth).
    Rewrite every path-like value to the file of the same name that really exists under `folder`,
    or null it out when no such file exists. Written next to the original as config_local.json."""
    cfg = json.loads(cfg_path.read_text(encoding="utf-8"))
    def fix(v, key=""):
        if isinstance(v, dict):
            return {k: fix(x, k) for k, x in v.items()}
        if isinstance(v, list):
            return [fix(x, key) for x in v]
        if isinstance(v, str) and re.search(r"\.(pth|pt|json|npy|pkl|txt)$", v) and ("/" in v or "\\" in v or key.endswith(("file", "path", "files"))):
            if Path(v).exists():
                return v
            hit = _find_file(folder, Path(v.replace("\\", "/")).name)
            if hit is not None:
                print(f"[Config] {key}: {v} -> {hit}")
                return str(hit)
            print(f"[Config] {key}: {v} not found, set to null")
            return None
        return v
    out = cfg_path.with_name("config_local.json")
    out.write_text(json.dumps(fix(cfg)), encoding="utf-8")
    return out

def create_synthesizer(lang: str) -> Synthesizer:
    folder = CHECKPOINTS / lang
    acoustic = folder / "fastpitch"
    vocoder = folder / "hifigan"
    speaker_file = _find_file(folder, "speakers.pth")
    print(f"[Loader] '{lang}' speakers.pth: {speaker_file or 'not found'}")

    return Synthesizer(
        tts_checkpoint=str(acoustic / "best_model.pth"),
        tts_config_path=str(_localize_config(acoustic / "config.json", folder)),
        tts_speakers_file=(str(speaker_file) if speaker_file else None),
        tts_languages_file=None,
        vocoder_checkpoint=str(vocoder / "best_model.pth"),
        vocoder_config=str(_localize_config(vocoder / "config.json", folder)),
        encoder_checkpoint="",
        encoder_config="",
        use_cuda=USE_CUDA,
    )

SPEAKER_PREFERENCE = ("female", "male")   # default pick when a checkpoint has several speakers
_speaker_names = {}                       # language -> list of speaker names in the checkpoint
WARMUP_TEXT = {"en": "Hello.", "hi": "नमस्ते।", "gu": "નમસ્તે."}

def detect_speakers(synth) -> list:
    try:
        manager = getattr(synth.tts_model, "speaker_manager", None)
        return list(getattr(manager, "name_to_id", {}) or {}) if manager is not None else []
    except Exception:
        return []

def choose_speaker(names, wanted=None):
    """None for single-speaker checkpoints; otherwise a valid speaker name."""
    if not names:
        return None
    if wanted:
        if wanted in names:
            return wanted
        raise ValueError(f"Unknown speaker '{wanted}'. Available: {names}")
    for pref in SPEAKER_PREFERENCE:
        for name in names:
            if pref in str(name).lower():
                return name
    return names[0]

def get_model(lang: str) -> Synthesizer:
    if lang in _models:
        return _models[lang]
    print(f"[Loader] Loading model for '{lang}' on {DEVICE}...")
    synthesizer = create_synthesizer(lang)
    _models[lang] = synthesizer
    _speaker_names[lang] = detect_speakers(synthesizer)
    print(f"[Loader] '{lang}' loaded. Speakers in checkpoint: {_speaker_names[lang] or 'single/none'}")
    return synthesizer

# --------------------------------------------------------------------------
# Text Preprocessing
# --------------------------------------------------------------------------
def normalize_text_for_tts(text: str, lang: str) -> str:
    clean = text.strip()
    # Replace excessive whitespaces and non-standard punctuation
    clean = re.sub(r"\s+", " ", clean)
    clean = clean.replace("\u200b", "") # Zero-width space
    clean = clean.replace("।", ".") # Replace Devanagari/Gujarati danda with period
    return clean

def split_into_sentences(text: str, max_chars: int = 180) -> list[str]:
    raw_sentences = [s.strip() for s in re.split(r"(?<=[.!?।\n])\s+", text) if s.strip()]
    chunks = []
    current = ""
    for s in raw_sentences:
        if not re.search(r"[\w\u0900-\u097F\u0A80-\u0AFF]", s):
            continue
        if not current:
            current = s
        elif len(current) + len(s) + 1 <= max_chars:
            current += " " + s
        else:
            chunks.append(current)
            current = s
    if current and re.search(r"[\w\u0900-\u097F\u0A80-\u0AFF]", current):
        chunks.append(current)
    return chunks

# --------------------------------------------------------------------------
# API Models & Endpoints
# --------------------------------------------------------------------------
class TTSRequest(BaseModel):
    text: str = Field(min_length=1, max_length=2000)
    language: str = Field(default="en", pattern="^(en|gu|hi)$")
    speaker: str | None = None   # optional, e.g. "male" or "female"; see /health for the names

def preload_all():
    """Load and warm up every language before the server starts answering, so no request pays the load time
    and any model problem shows up in the log immediately."""
    import gc
    for lang in LANGUAGES:
        try:
            synthesize_audio(WARMUP_TEXT[lang], lang)
            print(f"[Warmup] '{lang}' OK")
        except Exception as e:
            print(f"[Warmup Error] '{lang}': {type(e).__name__}: {e}")
        gc.collect()

@app.get("/")
@app.get("/health")
@app.get("/tts/health")
def health():
    return {
        "status": "ok",
        "service": "ai4bharat-indic-tts",
        "device": DEVICE,
        "languages": LANGUAGES,
        "loaded_models": list(_models.keys()),
        "speakers": _speaker_names,
    }

def synthesize_audio(text: str, lang: str, speaker: str | None = None) -> tuple[bytes, int]:
    cleaned = normalize_text_for_tts(text, lang)
    if not cleaned or not re.search(r"[\w\u0900-\u097F\u0A80-\u0AFF]", cleaned):
        raise ValueError("Text contains no speakable words")

    with _model_lock:
        model = get_model(lang)
        spk = choose_speaker(_speaker_names.get(lang, []), speaker)
        sr = model.output_sample_rate or 22050

        chunks = split_into_sentences(cleaned, max_chars=180)
        if not chunks:
            chunks = [cleaned]

        all_waveforms = []
        silence = np.zeros(int(sr * 0.25), dtype=np.float32)

        for chunk in chunks:
            if not re.search(r"[\w\u0900-\u097F\u0A80-\u0AFF]", chunk):
                continue
            try:
                audio = model.tts(chunk, speaker_name=spk) if spk is not None else model.tts(chunk)
                if audio is not None and len(audio) > 0:
                    waveform = np.asarray(audio, dtype=np.float32)
                    waveform = np.nan_to_num(waveform)
                    all_waveforms.append(waveform)
                    all_waveforms.append(silence)
            except Exception as e:
                print(f"[Synthesize Chunk Error] '{chunk[:40]}...': {e}")
                continue

        if not all_waveforms:
            raise RuntimeError("Model produced no audio for the input text")

        combined = np.concatenate(all_waveforms[:-1]) if len(all_waveforms) > 1 else all_waveforms[0]
        buf = io.BytesIO()
        sf.write(buf, combined, sr, format="WAV", subtype="PCM_16")
        return buf.getvalue(), sr

@app.post("/")
@app.post("/tts")
def generate_speech_binary(request: TTSRequest):
    """Returns raw WAV audio binary for direct playback or download."""
    try:
        wav_bytes, _ = synthesize_audio(request.text, request.language, request.speaker)
        return Response(
            content=wav_bytes,
            media_type="audio/wav",
            headers={"Content-Disposition": f'attachment; filename="speech_{request.language}.wav"'},
        )
    except ValueError as exc:
        raise HTTPException(status_code=400, detail=str(exc)) from exc
    except Exception as exc:
        raise HTTPException(status_code=500, detail=f"{type(exc).__name__}: {exc}") from exc

@app.post("/tts/synthesize")
@app.post("/synthesize")
def generate_speech_base64(request: TTSRequest):
    """Returns JSON payload with base64 encoded audio for microservice consumers."""
    try:
        wav_bytes, sr = synthesize_audio(request.text, request.language, request.speaker)
        b64 = base64.b64encode(wav_bytes).decode("utf-8")
        return {
            "audioContent": b64,
            "language": request.language,
            "sampleRate": sr,
            "format": "wav",
            "cached": False,
        }
    except ValueError as exc:
        raise HTTPException(status_code=400, detail=str(exc)) from exc
    except Exception as exc:
        raise HTTPException(status_code=500, detail=f"{type(exc).__name__}: {exc}") from exc

# Load and warm up all languages now (runs once, before uvicorn starts serving)
preload_all()

## 5. Install Cloudflare Tunnel


In [ ]:
cloudflared_path = "/content/cloudflared"
if not os.path.isfile(cloudflared_path):
    subprocess.run(["wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", cloudflared_path], check=True)
os.chmod(cloudflared_path, 0o755)
subprocess.run([cloudflared_path, "--version"], check=True)


## 6. Start Indic-TTS service on port 8004


In [ ]:
# Apply a hotpatch to transformers in the environment to resolve the import error:
# ImportError: cannot import name 'isin_mps_friendly' from 'transformers.pytorch_utils'
import sys
try:
    import transformers.pytorch_utils
    if not hasattr(transformers.pytorch_utils, "isin_mps_friendly"):
        # isin_mps_friendly is simply a fallback check; we can patch it to a standard comparison or dummy function
        def isin_mps_friendly(tensor, values):
            import torch
            return torch.isin(tensor, torch.tensor(values, device=tensor.device))
        transformers.pytorch_utils.isin_mps_friendly = isin_mps_friendly
        print("Successfully patched transformers.pytorch_utils.isin_mps_friendly!")
except ImportError:
    pass

def stop_matching(pattern):
    out = subprocess.run(["bash", "-lc", f"pgrep -f {pattern!r} || true"],
                         capture_output=True, text=True).stdout.split()
    for raw_pid in out:
        try:
            pid = int(raw_pid)
            if pid != os.getpid():
                os.kill(pid, signal.SIGTERM)
        except Exception:
            pass

stop_matching("uvicorn indic_tts_server:app")
time.sleep(2)

# Let's also patch the python process environment or rewrite the top of the server file
# to ensure the background subprocess doesn't crash on import.
server_file = "/content/drive/MyDrive/IndicTTSServer/indic_tts_server.py"
if os.path.exists(server_file):
    with open(server_file, "r") as f:
        content = f.read()
    # Inject the monkeypatch at the very top if it's not already there
    patch_code = """import sys
try:
    import transformers.pytorch_utils
    if not hasattr(transformers.pytorch_utils, 'isin_mps_friendly'):
        transformers.pytorch_utils.isin_mps_friendly = lambda tensor, values: tensor.cpu().apply_(lambda x: x in values).to(tensor.device)
except Exception:
    pass
"""
    if "isin_mps_friendly" not in content:
        with open(server_file, "w") as f:
            f.write(patch_code + "\n" + content)
        print("Injected transformers patch into server source file.")

tts_log = "/content/indic_tts.log"
tts_log_handle = open(tts_log, "w")
tts_proc = subprocess.Popen(
    ["uvicorn", "indic_tts_server:app", "--host", "0.0.0.0", "--port", "8004"],
    cwd=TTS_SERVER_DIR, stdout=tts_log_handle, stderr=subprocess.STDOUT
)

# Wait for service readiness
start_time = time.time()
ready = False
print("Starting Indic-TTS service...")
while time.time() - start_time < 900:
    if tts_proc.poll() is not None:
        print("TTS process exited prematurely:", tts_proc.returncode)
        print(open(tts_log, errors="replace").read()[-4000:])
        break
    try:
        r = requests.get("http://127.0.0.1:8004/health", timeout=3)
        if r.status_code == 200:
            print(f"Indic-TTS service ready in {time.time()-start_time:.1f}s:", r.json())
            ready = True
            break
    except requests.RequestException:
        pass
    time.sleep(3)

assert ready, "Indic-TTS failed to start. Inspect /content/indic_tts.log."
print("\nStartup log (look for [Warmup] lines):")
print(open(tts_log, errors="replace").read()[-3000:])

In [ ]:
local_cases = [
    ("gu", "નમસ્તે ખેડૂત મિત્ર! સ્માર્ટ ફાર્મિંગમાં તમારું સ્વાગત છે."),
    ("hi", "नमस्ते किसान भाई! स्मार्ट फार्मिंग में आपका स्वागत है।"),
    ("en", "Hello farmer! Welcome to Smart Farming."),
]
print("health:", requests.get("http://127.0.0.1:8004/health", timeout=10).json())
for lang, text in local_cases:
    t0 = time.time()
    try:
        r = requests.post("http://127.0.0.1:8004/tts", json={"text": text, "language": lang}, timeout=600)
        if r.status_code == 200:
            print(f"{lang}: OK in {time.time() - t0:.1f}s, {len(r.content):,} bytes")
        else:
            print(f"{lang}: HTTP {r.status_code} -> {r.text[:500]}")
    except Exception as exc:
        print(f"{lang}: FAILED after {time.time() - t0:.1f}s: {exc}")

## 7. Create Quick Tunnel and update Cloudflare KV (`TTS_URL`)


In [ ]:
def open_tunnel(port, timeout=90):
    log_path = f"/content/tunnel_{port}.log"
    log_handle = open(log_path, "w")
    proc = subprocess.Popen(
        ["/content/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}", "--no-autoupdate"],
        stdout=log_handle, stderr=subprocess.STDOUT
    )
    deadline = time.time() + timeout
    while time.time() < deadline:
        content = open(log_path, errors="replace").read()
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
        if match:
            return proc, match.group(0), log_path
        if proc.poll() is not None:
            break
        time.sleep(1)
    raise RuntimeError(f"Could not obtain tunnel URL for port {port}. Log: {log_path}")

tts_tunnel_proc, TTS_URL, tts_tunnel_log = open_tunnel(8004)
print("TTS_URL =", TTS_URL)

with open("/content/drive/MyDrive/service_urls_tts.json", "w", encoding="utf-8") as f:
    json.dump({"tts": TTS_URL}, f, indent=2)

import requests
from google.colab import userdata

def colab_secret(name, required=True):
    value = userdata.get(name)
    if required and not value:
        raise RuntimeError(f"Missing Colab Secret {name}. Add it to Secrets and enable access.")
    return value

CF_API_TOKEN = colab_secret("CF_API_TOKEN")
CF_ACCOUNT_ID = colab_secret("CF_ACCOUNT_ID")
CF_KV_NAMESPACE_ID = colab_secret("CF_KV_NAMESPACE_ID")

def update_cloudflare_kv(key, value):
    """Update Cloudflare Worker KV storage key."""
    endpoint = f"https://api.cloudflare.com/client/v4/accounts/{CF_ACCOUNT_ID}/storage/kv/namespaces/{CF_KV_NAMESPACE_ID}/values/{key}"
    response = requests.put(
        endpoint,
        headers={"Authorization": f"Bearer {CF_API_TOKEN}", "Content-Type": "text/plain"},
        data=value.encode("utf-8"),
        timeout=30
    )
    try:
        payload = response.json()
    except Exception:
        payload = {}
    if not response.ok or payload.get("success") is False:
        raise RuntimeError(f"Cloudflare KV update failed for {key}: {response.text[:400]}")
    print(f"[Cloudflare KV] Updated {key} -> {value}")

def wait_for_tunnel(url, timeout=180):
    """A new quick tunnel prints its URL before it is connected; wait until it really answers."""
    end = time.time() + timeout
    while time.time() < end:
        try:
            if requests.get(f"{url}/health", timeout=10).status_code == 200:
                return True
        except requests.RequestException:
            pass
        time.sleep(3)
    return False

print("Waiting for the tunnel to accept connections...")
assert wait_for_tunnel(TTS_URL), f"Tunnel {TTS_URL} never answered. Check {tts_tunnel_log}."
print("Tunnel is live.")

update_cloudflare_kv("TTS_URL", TTS_URL)

GATEWAY_BASE = colab_secret("CF_WORKER_BASE_URL").rstrip("/")
print("\nStable gateway base URL:", GATEWAY_BASE)
try:
    gw_status = requests.get(f"{GATEWAY_BASE}/gateway/status", timeout=20).json()
    print("Gateway status:", json.dumps(gw_status))
    if "kv_binding_missing" in json.dumps(gw_status.get("routes", {})):
        print("\n>>> The Worker has no KV binding named UPSTREAMS. Cloudflare dashboard > Workers & Pages > "
              "your Worker > Settings > Bindings > Add > KV namespace > variable name UPSTREAMS > select the "
              "namespace that matches CF_KV_NAMESPACE_ID > Deploy. Then run this cell again.")
    if not gw_status.get("has_auth_key"):
        print("\n>>> The Worker has no GATEWAY_API_KEY secret. Add it under Settings > Variables and Secrets.")
except Exception as exc:
    print("Could not read gateway status:", exc)
print("TTS route: POST {gateway}/tts/tts (WAV) or /tts/synthesize (base64 JSON), header Authorization: Bearer GATEWAY_API_KEY")

## 8. Test speech synthesis for Gujarati, Hindi, and English

Generates speech through the stable Cloudflare Worker Gateway and verifies audio output directly inside the notebook.


In [ ]:
import IPython.display as ipd
from google.colab import userdata

GATEWAY_BASE = (userdata.get("CF_WORKER_BASE_URL") or "").rstrip("/")
GATEWAY_KEY = userdata.get("GATEWAY_API_KEY") or ""
assert GATEWAY_BASE and GATEWAY_KEY, "Colab Secrets CF_WORKER_BASE_URL and GATEWAY_API_KEY required."
auth_headers = {"Authorization": f"Bearer {GATEWAY_KEY}", "Content-Type": "application/json"}

def explain(resp):
    body = resp.text[:300]
    if resp.status_code == 401:
        return "Gateway key mismatch: GATEWAY_API_KEY in Colab Secrets must equal the Worker secret."
    if resp.status_code == 500 and "UPSTREAMS" in body:
        return "The Worker has no KV binding named UPSTREAMS (Worker > Settings > Bindings > Add > KV namespace)."
    if resp.status_code == 503:
        return "TTS_URL is not set in the KV namespace the Worker is bound to. Check CF_KV_NAMESPACE_ID."
    if resp.status_code in (502, 530):
        return "Tunnel not reachable yet, or KV still serves the previous tunnel URL (KV can take about 60 s to update). Waiting..."
    return body

# 1. Wait until the gateway forwards to the tunnel
ok, deadline = False, time.time() + 180
while time.time() < deadline:
    try:
        r = requests.get(f"{GATEWAY_BASE}/tts/health", headers=auth_headers, timeout=30)
    except requests.RequestException as exc:
        print("gateway request failed:", exc); time.sleep(5); continue
    if r.status_code == 200:
        print("Gateway -> TTS health OK:", r.json()); ok = True; break
    print(f"gateway /tts/health -> HTTP {r.status_code}: {explain(r)}")
    if r.status_code in (401, 503) or (r.status_code == 500 and "UPSTREAMS" in r.text):
        break
    time.sleep(10)
assert ok, "The gateway cannot reach the TTS service yet. Fix the problem printed above and run this cell again."

# 2. Synthesize through the gateway
test_cases = [
    ("gu", "નમસ્તે ખેડૂત મિત્ર! સ્માર્ટ ફાર્મિંગમાં તમારું સ્વાગત છે. તમારા પાકનું આરોગ્ય સારું છે."),
    ("hi", "नमस्ते किसान भाई! स्मार्ट फार्मिंग में आपका स्वागत है। आपकी फसल स्वस्थ है।"),
    ("en", "Hello farmer! Welcome to Smart Farming. Your crop health looks great today."),
]
for lang, text in test_cases:
    print(f"\n[Synthesizing {lang.upper()}] {text}")
    t0 = time.time()
    try:
        resp = requests.post(f"{GATEWAY_BASE}/tts/tts", headers=auth_headers,
                             json={"text": text, "language": lang}, timeout=240)
    except Exception as exc:
        print(f"FAILED after {time.time() - t0:.1f}s: {exc}")
        continue
    if resp.status_code == 200:
        print(f"Success in {time.time() - t0:.2f}s, audio size {len(resp.content):,} bytes")
        with open(f"/content/output_{lang}.wav", "wb") as f:
            f.write(resp.content)
        ipd.display(ipd.Audio(resp.content, autoplay=False))
    else:
        print(f"Failed (HTTP {resp.status_code}): {explain(resp)}")

## 9. Diagnostics


In [ ]:
for log_path in ["/content/indic_tts.log", "/content/tunnel_8004.log"]:
    print("\n====", log_path, "====")
    if os.path.isfile(log_path):
        print(open(log_path, errors="replace").read()[-3000:])
    else:
        print("Not found")
print("\nGPU Status:")
subprocess.run(["bash", "-lc", "nvidia-smi || echo 'Running in CPU mode.'"])
